In [2]:
import pandas as pd 

In [3]:
df = pd.read_csv("../data/raw/HouseTS.csv")

In [13]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [14]:
df.shape

(884092, 39)

In [16]:
df.head(2)

,date,median_sale_price,median_list_price,median_ppsf,median_list_ppsf,homes_sold,pending_sales,new_listings,inventory,median_dom,avg_sale_to_list,sold_above_list,off_market_in_two_weeks,city,zipcode,year,bank,bus,hospital,mall,park,restaurant,school,station,supermarket,Total Population,Median Age,Per Capita Income,Total Families Below Poverty,Total Housing Units,Median Rent,Median Home Value,Total Labor Force,Unemployed Population,Total School Age Population,Total School Enrollment,Median Commute Time,price,city_full
0,2012-03-31,46550.0,217450.0,31.813674,110.183666,14.0,23.0,44.0,64.0,59.5,0.943662,0.142857,0.043478,ATL,30002,2012,12.0,2.0,4.0,1.0,60.0,45.0,57.0,4.0,7.0,5811.0,36.3,33052.0,5811.0,2677.0,710.0,279500.0,3171.0,460.0,5408.0,5408.0,2492.0,200773.999557,Atlanta-Sandy Springs-Alpharetta
1,2012-04-30,61870.0,245000.0,40.723982,130.528256,22.0,29.0,56.0,69.0,89.5,0.946642,0.090909,0.034483,ATL,30002,2012,12.0,2.0,4.0,1.0,60.0,45.0,57.0,4.0,7.0,5811.0,36.3,33052.0,5811.0,2677.0,710.0,279500.0,3171.0,460.0,5408.0,5408.0,2492.0,202421.064584,Atlanta-Sandy Springs-Alpharetta


In [4]:
df['date'] = pd.to_datetime(df['date'])
print(df['date'].dt.year.value_counts())

date
2013    74712
2014    74712
2015    74712
2016    74712
2017    74712
2018    74712
2019    74712
2020    74712
2021    74712
2022    74712
2023    74712
2012    62260
Name: count, dtype: int64


Avoid Leakage
Your train set (2012–2019) is “clean” historical training.

Your eval set (2020–2021) is “unseen” during training, but still used for validation and hyperparameter tuning.

Your holdout set (2022–2023) is completely untouched until the very end.

In [ ]:
# sort by date 
df = df.sort_values("date")

,date,median_sale_price,median_list_price,median_ppsf,median_list_ppsf,homes_sold,pending_sales,new_listings,inventory,median_dom,...,Total Housing Units,Median Rent,Median Home Value,Total Labor Force,Unemployed Population,Total School Age Population,Total School Enrollment,Median Commute Time,price,city_full
0,2012-03-31,46550.0,217450.0,31.813674,110.183666,14.0,23.0,44.0,64.0,59.5,...,2677.0,710.0,279500.0,3171.0,460.0,5408.0,5408.0,2492.0,200773.999557,Atlanta-Sandy Springs-Alpharetta
183322,2012-03-31,156500.0,182188.5,119.789171,140.818182,177.0,0.0,260.0,310.0,74.0,...,20816.0,747.0,163000.0,24348.0,2632.0,39826.0,39826.0,20345.0,137474.419588,Charlotte-Concord-Gastonia
272498,2012-03-31,179000.0,210000.0,97.687880,103.876898,62.0,100.0,107.0,50.0,42.5,...,13318.0,1099.0,233000.0,19023.0,1253.0,28622.0,28622.0,16400.0,207284.065948,Denver-Aurora-Lakewood
272640,2012-03-31,150000.0,193950.0,66.799886,73.878756,141.0,210.0,212.0,129.0,43.0,...,14240.0,874.0,170800.0,20029.0,2133.0,38001.0,38001.0,17057.0,151368.875848,Denver-Aurora-Lakewood
244950,2012-03-31,635000.0,785000.0,358.716732,409.102025,69.0,83.0,150.0,123.0,33.5,...,14196.0,1801.0,873700.0,16643.0,777.0,29093.0,29093.0,14038.0,861060.378768,DC_Metro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13631,2023-12-31,309995.0,334900.0,167.288557,170.299728,164.0,158.0,222.0,178.0,54.0,...,15336.0,1334.0,228500.0,21716.0,1444.0,41062.0,41062.0,17948.0,316893.196742,Atlanta-Sandy Springs-Alpharetta
22151,2023-12-31,250625.0,319000.0,154.791209,182.718894,82.0,93.0,133.0,117.0,51.0,...,17193.0,1026.0,220400.0,15766.0,1363.0,32226.0,32226.0,12141.0,257756.335374,Atlanta-Sandy Springs-Alpharetta
23429,2023-12-31,544400.0,534500.0,292.148843,294.015702,96.0,89.0,141.0,122.0,39.5,...,18013.0,1665.0,496000.0,20807.0,760.0,28870.0,28870.0,15876.0,497563.715446,Atlanta-Sandy Springs-Alpharetta
5963,2023-12-31,625000.0,556000.0,224.481743,226.628895,58.0,55.0,65.0,39.0,21.5,...,13257.0,1542.0,510700.0,17380.0,724.0,32958.0,32958.0,12065.0,633756.634573,Atlanta-Sandy Springs-Alpharetta


In [12]:

# Define cutoff dates
cutoff_date_eval = "2020-01-01"   # validation starts
cutoff_date_holdout = "2022-01-01"  # holdout starts

In [13]:
# Train: before 2020
train_df = df[df["date"] < cutoff_date_eval]

# Validation/Eval: 2020–2021
eval_df = df[(df["date"] >= cutoff_date_eval) & (df["date"] < cutoff_date_holdout)]

# Holdout: 2022–2023
holdout_df = df[df["date"] >= cutoff_date_holdout]

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)
print("Holdout shape:", holdout_df.shape)


Train shape: (585244, 39)
Eval shape: (149424, 39)
Holdout shape: (149424, 39)


In [14]:
train_df.head(2)

,date,median_sale_price,median_list_price,median_ppsf,median_list_ppsf,homes_sold,pending_sales,new_listings,inventory,median_dom,...,Total Housing Units,Median Rent,Median Home Value,Total Labor Force,Unemployed Population,Total School Age Population,Total School Enrollment,Median Commute Time,price,city_full
0,2012-03-31,46550.0,217450.0,31.813674,110.183666,14.0,23.0,44.0,64.0,59.5,...,2677.0,710.0,279500.0,3171.0,460.0,5408.0,5408.0,2492.0,200773.999557,Atlanta-Sandy Springs-Alpharetta
183322,2012-03-31,156500.0,182188.5,119.789171,140.818182,177.0,0.0,260.0,310.0,74.0,...,20816.0,747.0,163000.0,24348.0,2632.0,39826.0,39826.0,20345.0,137474.419588,Charlotte-Concord-Gastonia


In [15]:
eval_df.head(2)

,date,median_sale_price,median_list_price,median_ppsf,median_list_ppsf,homes_sold,pending_sales,new_listings,inventory,median_dom,...,Total Housing Units,Median Rent,Median Home Value,Total Labor Force,Unemployed Population,Total School Age Population,Total School Enrollment,Median Commute Time,price,city_full
272734,2020-01-31,391000.0,399250.0,144.336169,149.398137,298.0,289.0,218.0,100.0,34.0,...,15446.0,1221.0,305900.0,26080.0,1267.0,49042.0,49042.0,23285.0,3.580094e+05,Denver-Aurora-Lakewood
580874,2020-01-31,12112500.0,10725000.0,1571.709470,2947.265773,2.0,2.0,2.0,3.0,196.5,...,952.0,1666.0,2000001.0,111.0,0.0,315.0,315.0,80.0,3.729513e+06,New York-Newark-Jersey City


In [16]:
holdout_df.head(2)

,date,median_sale_price,median_list_price,median_ppsf,median_list_ppsf,homes_sold,pending_sales,new_listings,inventory,median_dom,...,Total Housing Units,Median Rent,Median Home Value,Total Labor Force,Unemployed Population,Total School Age Population,Total School Enrollment,Median Commute Time,price,city_full
493994,2022-01-31,590000.0,657450.0,343.870968,301.327241,19.0,10.0,6.0,4.0,48.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.934719e+05,New York-Newark-Jersey City
835362,2022-01-31,1400000.0,1522500.0,857.142857,891.865994,33.0,34.0,28.0,11.0,28.0,...,6463.0,2443.0,1173700.0,6859.0,371.0,11565.0,11565.0,4613.0,1.582472e+06,San Francisco-Oakland-Berkeley


In [18]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

train_df.to_csv(RAW_DATA_DIR / "train.csv", index=False)
eval_df.to_csv(RAW_DATA_DIR / "eval.csv", index=False)
holdout_df.to_csv(RAW_DATA_DIR / "holdout.csv", index=False)